# HStream Extractor (Google Colab)

Bulk downloader + optional subtitle remuxer for [hstream.moe](https://hstream.moe).

**Features**
- yt-dlp + aria2c downloads
- Cookie support for blocked / age-gated titles
- Optional external `.ass` subtitle download + MKV remux
- Progress bars

> **Never paste real cookies into a public notebook.**  
> Upload a `cookies.txt` file instead.

## 1. Install dependencies

In [ ]:
!pip install -q --upgrade yt-dlp requests tqdm
!apt-get update -qq
!apt-get install -y -qq aria2 ffmpeg > /dev/null
print('Dependencies installed')

## 2. Configuration

Edit the values in the next cell:
- `URL_LIST` — space-separated URLs
- `COOKIES_FILE` — path to uploaded cookies.txt (leave empty if not needed)
- `SERIES_SLUG` — optional, use dots e.g. `Sweet.Home.H.na.Oneesan.wa.Suki.desu.ka`

For blocked titles: log in to hstream.moe, export cookies (Netscape format), upload `cookies.txt` via the left sidebar.

In [ ]:
# ===== SETTINGS (edit these) =====
URL_LIST = "https://hstream.moe/hentai/example-1 https://hstream.moe/hentai/example-2"
DESTINATION_FOLDER = "/content/downloads"
COOKIES_FILE = ""          # e.g. "cookies.txt" after uploading
SERIES_SLUG = ""           # optional, dots instead of hyphens
YEAR = "2024"
# =================================

print("Settings loaded.")
print("URLs:", URL_LIST)

## 3. Run the extractor

In [ ]:
import re
import subprocess
from pathlib import Path
import requests
from tqdm.notebook import tqdm

dest = Path(DESTINATION_FOLDER)
dest.mkdir(parents=True, exist_ok=True)

urls = [u.strip() for u in URL_LIST.replace("\n", " ").split() if u.strip()]
print(f"{len(urls)} URL(s) -> {dest}\n")

cookies_path = Path(COOKIES_FILE) if COOKIES_FILE.strip() else None
if cookies_path and not cookies_path.exists():
    print(f"WARNING: Cookies file not found: {cookies_path}")
    print("Upload it via the left sidebar or leave COOKIES_FILE empty.")

def download_video(url):
    output_template = str(dest / "%(title)s.%(ext)s")
    cmd = [
        "yt-dlp",
        "--downloader", "aria2c",
        "--downloader-args", "aria2c:-x 16 -s 16 -k 1M",
        "--concurrent-fragments", "8",
        "-o", output_template,
        "--no-mtime",
    ]
    if cookies_path and cookies_path.exists():
        cmd.extend(["--cookies", str(cookies_path)])
    cmd.append(url)
    print(f"Downloading: {url}")
    subprocess.run(cmd, check=True)
    files = list(dest.glob("*"))
    if not files:
        raise FileNotFoundError("No file downloaded")
    return max(files, key=lambda p: p.stat().st_ctime)

def download_subtitle(sub_url, sub_path):
    try:
        with requests.get(sub_url, stream=True, timeout=30) as r:
            if r.status_code != 200:
                return False
            total = int(r.headers.get("content-length", 0))
            with open(sub_path, "wb") as f, tqdm(
                total=total, unit="B", unit_scale=True, unit_divisor=1024, leave=False
            ) as bar:
                for chunk in r.iter_content(8192):
                    if chunk:
                        f.write(chunk)
                        bar.update(len(chunk))
        return True
    except Exception:
        return False

def remux(video, sub, out):
    subprocess.run([
        "ffmpeg", "-y", "-i", str(video), "-i", str(sub),
        "-map", "0", "-map", "1", "-c", "copy",
        "-metadata:s:s:0", "language=eng", str(out)
    ], check=True, capture_output=True)

for i, url in enumerate(tqdm(urls, desc="Overall", unit="video"), 1):
    print(f"\n[{i}/{len(urls)}] {url}")
    try:
        video_path = download_video(url)
        base = video_path.stem
        final_mkv = dest / f"{base}.mkv"

        ep_match = re.search(r"-(\d+)/?$", url.rstrip("/"))
        if not ep_match:
            print("No episode number found - keeping original")
            continue

        ep_num = int(ep_match.group(1))
        slug = SERIES_SLUG.strip() or re.sub(r"-\d+$", "", url.rstrip("/").split("/")[-1])
        sub_url = f"https://oppai-str.shoujo-h.org/{YEAR}/{slug}/E{ep_num:02d}/eng.ass"
        sub_path = dest / f"{base}.ass"

        if download_subtitle(sub_url, sub_path):
            print("Remuxing...")
            remux(video_path, sub_path, final_mkv)
            sub_path.unlink(missing_ok=True)
            if video_path != final_mkv and video_path.exists():
                video_path.unlink()
            print(f"OK: {final_mkv}")
        else:
            print(f"Subtitle not found - kept {video_path}")
    except Exception as e:
        print(f"Failed: {e}")

print("\nDone! Files are in:", dest)

## 4. Download your files

Download files from the left sidebar (Files), or zip everything:

In [ ]:
!zip -r /content/hstream_downloads.zip {DESTINATION_FOLDER}
print("Created: /content/hstream_downloads.zip")
print("Download it from the left sidebar -> Files")

---
Repo: [Hstream-Extractor](https://github.com/zenin-373/Hstream-Extractor)